<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/03-dataframe-fundamentals/05-null-handling.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)

print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))


# 3.5 — Null handling

The 13 + 25 ≠ 41 demo is the heart of this notebook.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, DateType,
)

spark = (
    SparkSession.builder
    .appName("3.5-null-handling")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

ORDERS_SCHEMA = StructType([
    StructField("order_id",    IntegerType(), False),
    StructField("customer_id", StringType(),  False),
    StructField("order_date",  DateType(),    True),
    StructField("product",     StringType(),  True),
    StructField("category",    StringType(),  True),
    StructField("quantity",    IntegerType(), True),
    StructField("unit_price",  DoubleType(),  True),
    StructField("country",     StringType(),  True),
])
orders = (
    spark.read.option("header", True)
    .schema(ORDERS_SCHEMA)
    .csv(f"{DATA_DIR}/orders.csv")
)

## The disappearing rows: a filter and its negation don't partition the data

In [ ]:
total = orders.count()
eq    = orders.filter(col("country") == "IN").count()
neq   = orders.filter(col("country") != "IN").count()
print(f"total={total}   ==IN: {eq}   !=IN: {neq}   sum: {eq + neq}")
print(f"vanished from BOTH branches: {total - eq - neq}")

In [ ]:
# Why: the comparison yields NULL, not False — see it directly
orders.select(
    "order_id", "country",
    (col("country") == "IN").alias("eq_IN"),
    (col("country") != "IN").alias("neq_IN"),
).filter(col("country").isNull()).show()

In [ ]:
# The explicit fix — and null-safe equality
fixed = orders.filter((col("country") != "IN") | col("country").isNull()).count()
print("explicit null branch:", eq + fixed, "== 41")

orders.select("country", col("country").eqNullSafe("IN").alias("eqns")) \
    .filter(col("country").isNull()).show(1)   # false, not null

## dropna — always with subset, always counting the damage

In [ ]:
print("drop any-null row:      ", orders.na.drop().count(), "of", total)
print("drop on quantity only:  ", orders.na.drop(subset=["quantity"]).count())

before = orders.count()
clean = orders.na.drop(subset=["quantity", "unit_price"])
print(f"dropped {before - clean.count()} rows lacking quantity/price")

## fillna — dict form, and the silent type-mismatch

In [ ]:
filled = orders.na.fill({"quantity": 0, "country": "??"})
filled.filter((col("quantity") == 0) | (col("country") == "??")) \
    .select("order_id", "quantity", "country").show()

# Type mismatch does NOTHING, silently: fill a string column with an int
still_null = orders.na.fill(0, subset=["country"]).filter(col("country").isNull()).count()
print("country nulls after fill(0):", still_null, "<- unchanged, no error")

## F.coalesce (first non-null) and F.nullif (sentinel -> null)

In [ ]:
orders.select(
    "order_id", "quantity",
    F.coalesce(col("quantity"), F.lit(1)).alias("qty_final"),
).filter(col("quantity").isNull()).show()

# nullif: turn a sentinel into a real null
sensors = spark.createDataFrame([(21.5,), (-999.0,), (19.8,)], "temp DOUBLE")
sensors.select("temp", F.nullif(col("temp"), F.lit(-999.0)).alias("temp_clean")).show()

## Aggregates skip nulls — and groupBy keeps them as a group

In [ ]:
orders.agg(
    F.count("*").alias("rows"),
    F.count("quantity").alias("qty_values"),
    F.round(F.avg("quantity"), 3).alias("avg_skips_nulls"),
).show()

orders.groupBy("country").count().orderBy("country").show()   # null is a row here

## Exercises

1. Average `quantity` two ways: skipping nulls (default) and treating null as 0 (`coalesce` first). Which is 'correct' if null means 'quantity not recorded'? If it means 'no items shipped'?
2. Revenue per country where unknown country appears as its own `"UNKNOWN"` bucket — using `fillna` before the groupBy. Then the same with `F.coalesce` inside the groupBy. Same result?
3. Build the complete-partition check as a reusable assertion: for any condition `c`, verify `filter(c).count() + filter(~c).count() + filter(c.isNull()).count() == df.count()`. Test it on `quantity >= 2`.
4. `sensors` above: compute the average temperature with and without the sentinel cleanup. How wrong was the naive number?

In [ ]:
# your exercise code here
